In [1]:
import json

import pandas as pd
from tqdm import tqdm
from jsonlines import jsonlines

from orqa.utils import sanitize_string

In [2]:
data_path                   = '../data'
tmp_path                    = f'{data_path}/tmp'

# the tables metadata obtained from the Open Data Canada 
# filtering by date, around July 2023
snapshot_path               = f'{data_path}/snapshot.jsonl'

# the above metadata enriched with the relative table schema
snapshot_with_schema_path   = f'{data_path}/snapshot_with_schema_also_for_zip.jsonl'

In [24]:
with jsonlines.open(snapshot_path) as reader:
    metadata = list(reader.iter())

In [25]:
print(f'Number of tables in the snapshot: {len(metadata)}')

Number of tables in the snapshot: 27752


In [6]:
[md['url'] for md in metadata[:10]]

['https://www.canada.ca/content/dam/cra-arc/migration/cra-arc/gncy/stts/gb10/pst/fnl/csv/t01ca.csv',
 'https://www.canada.ca/content/dam/eccc/migration/main/indicateurs-indicators/78a7ae2b-0d6c-4690-bab0-57652ebca282/globaltrendsbirdspeciessurvival_en.csv',
 'https://www.canada.ca/content/dam/cra-arc/migration/cra-arc/gncy/stts/opendata/bnfts/gdlntbls/cctbncbs-pfcespne-2013-eng.csv',
 'https://www150.statcan.gc.ca/n1/tbl/csv/33100655-fra.zip',
 'https://www150.statcan.gc.ca/n1/tbl/csv/33100633-fra.zip',
 'https://www150.statcan.gc.ca/n1/tbl/csv/33100653-fra.zip',
 'https://open.canada.ca/data/en/dataset/79e83e09-be72-4cec-b410-1e69aaa450f7/resource/7511b1a3-48c7-4386-ae6e-f2c4e87c6e84/download/2013-14_perchlorate_in_selected_foods_perchlorate-dans-certains-aliments.csv',
 'https://www12.statcan.gc.ca/health-sante/82-228/details/download-telecharger/all_geos/final/download.cfm?Lang=E&Format=CSV',
 'https://www150.statcan.gc.ca/n1/tbl/csv/98100289-fra.zip',
 'https://www150.statcan.gc.ca

In [3]:
with jsonlines.open(snapshot_with_schema_path) as reader:
    metadata_with_schema = list(reader.iter())

In [4]:
print(f'Number of resources which have provided a schema: {len([csv_md for csv_md in metadata_with_schema if csv_md['schema']])}')

Number of resources which have provided a schema: 10


In [28]:
len(metadata_with_schema)

100

In [6]:
[csv_md for csv_md in metadata_with_schema if csv_md['schema']]

[{'id': '383d565c-fa77-411e-9717-0e8519926ba3',
  'package_id': 'b8964f66-ce40-4088-9209-1c29f15a345d',
  'url': 'https://data-donnees.ec.gc.ca/data/substances/monitor/impact-of-municipal-effluents-and-hydrological-regime-on-myxozoan-parasite-communities-of-fish/SLAP_SpottailShiner_MyxozoanCommunities_EN_FR.csv',
  'format': 'CSV',
  'description': '',
  'hash': '',
  'position': 0,
  'name': 'SLAP SpottailShiner MyxozoanCommunities EN_FR.csv',
  'resource_type': 'dataset',
  'mimetype': None,
  'mimetype_inner': None,
  'size': None,
  'created': '2019-01-24T15:13:49.976218',
  'last_modified': None,
  'metadata_modified': '2019-01-24T15:13:49.976218',
  'cache_url': None,
  'cache_last_updated': None,
  'url_type': None,
  'state': 'active',
  'name_translated': '{"fr": "SLAP SpottailShiner MyxozoanCommunities EN_FR.csv", "en": "SLAP SpottailShiner MyxozoanCommunities EN_FR.csv"}',
  'language': '["en", "fr"]',
  'unique_identifier': '',
  'date_published': '',
  'datastore_contains_

In [ ]:
for csv_md in metadata:
    if 'schema' not in csv_md:
        csv_md['schema'] = []

### Load the Clusters created from the Tables zipped into LakeBench

In [11]:
join_clusters_path  = f'{data_path}/join/clusters/clusters_CAN.json'
union_clusters_path = f'{data_path}/union/clusters/clusters_CAN.json'

join_schema_path    = f'{data_path}/join/schema/tables_schema_CAN.json'
union_schema_path   = f'{data_path}/union/schema/tables_schema_CAN.json' 

with open(join_clusters_path) as fr:
    join_clusters = json.load(fr)

with open(union_clusters_path) as fr:
    union_clusters = json.load(fr)

with open(join_schema_path) as fr:
    join_tables_schema = json.load(fr)

with open(union_schema_path) as fr:
    union_tables_schema = json.load(fr)

In [12]:
from itertools import chain


join_all_attributes = set(chain(*join_tables_schema.values()))
union_all_attributes = set(chain(*union_tables_schema.values()))
len(join_all_attributes), len(union_all_attributes)

(1336, 2815)

In [ ]:
# keep only tables which have a not-empty schema
metadata = list(filter(lambda md: md['schema'], metadata))
len(metadata)

13566

In [ ]:
# clean the schema of the tables from the snapshot
for md in metadata:
    md['schema'] = list(map(sanitize_string, md['schema']))

In [19]:
# takes the tables whose schema is fully contained into the join or union macrosets
tables_in_join  = list(filter(lambda md: set(md['schema']).intersection(join_all_attributes) == set(md['schema']), metadata))
tables_in_union = list(filter(lambda md: set(md['schema']).intersection(union_all_attributes) == set(md['schema']), metadata))

len(tables_in_join), len(tables_in_union)

(138, 204)

In [20]:
# we can focus only on the tables in the union case
# because the attributes from the union tables seem to be a superset of the join case
len(set(md['id'] for md in tables_in_join).difference(set(md['id'] for md in tables_in_union)))

0

In [21]:
tables_in_union[:1]

[{'id': '1d12079c-ae9b-49f1-bd65-56a5dc984673',
  'package_id': '53f8ab93-ab62-486a-b563-1381224fcbc7',
  'url': 'https://ised-isde.canada.ca/site/canadian-importers-database/sites/default/files/attachments/HS10Description2019.csv',
  'format': 'CSV',
  'description': '',
  'hash': '',
  'position': 0,
  'name': 'HS10 Description',
  'resource_type': 'dataset',
  'mimetype': None,
  'mimetype_inner': None,
  'size': None,
  'created': '2020-12-07T21:50:49.781530',
  'last_modified': None,
  'metadata_modified': '2020-12-07T21:50:49.781530',
  'cache_url': None,
  'cache_last_updated': None,
  'url_type': None,
  'state': 'active',
  'name_translated': '{"fr": "Description SH10", "en": "HS10 Description"}',
  'language': '["en", "fr"]',
  'date_published': '',
  'unique_identifier': '',
  'validation_timestamp': '',
  'datastore_contains_all_records_of_source_file': False,
  'validation_status': '',
  'datastore_active': False,
  'character_set': '',
  'data_quality': '[]',
  'schema': 

In [22]:
union_clusters

{'max_num_of_cooccurrences': 1,
 'num_perm': 16,
 'l': 4,
 'k': 200,
 'clusters': [{'key': 0,
   'schema': ['sector',
    'type_of_power_plant',
    'canadian_portfolio_securities',
    'type_of_employee',
    'service_detail',
    'sector_accounts',
    'institution',
    'number_of_owners',
    'period_of_construction',
    'release',
    'type_of_case',
    'jurisdiction',
    'index',
    'payment_collection_rate',
    'inputs_outputs',
    'stormwater_assets',
    'mobility_indicators',
    'measures',
    'age_of_victim',
    'seasonality',
    'personal_protective_equipment_or_supplies',
    'interjurisdictional_support_order_status',
    'owner_characteristics_at_the_property_level',
    'commodity',
    'fuel_type',
    'machinery_and_equipment,_domestic_and_imported',
    'incorporation_status',
    'statistics',
    'overtime',
    'products',
    'import_and_export_major_groups_and_standard_international_trade_classification_(sitc_revision_3)',
    'credit_liabilities_of_pr

In [26]:
from collections import defaultdict


candidates = defaultdict(list)

# for each snapshot table, explore each cluster:
# if there is any cluster table which has an identical schema, save it
for md in tqdm(tables_in_union):
    schema = set(md['schema'])
    url = md['url']
    tid = md['id']

    for cluster in tqdm(union_clusters['clusters'], position=1, leave=False):
        if not schema.intersection(cluster['schema']) == schema:
            continue
        for csv_id in cluster['ids']:
            if schema == set(union_tables_schema[csv_id]):
                candidates[tid].append(csv_id)

100%|██████████| 204/204 [00:00<00:00, 315.32it/s]


In [27]:
remote_table_ids = list(candidates.keys())
len(remote_table_ids)

139

In [28]:
tid = remote_table_ids[138]
candidates[tid]

['CAN_CSV0000000000004354.csv', 'CAN_CSV0000000000016880.csv']

In [29]:
md = [t for t in tables_in_union if t['id'] == tid][0]
md

{'id': '6095d1b8-ebbb-46d2-9b47-5b6d64f8d4ed',
 'package_id': 'f7371bb6-a7f7-49e7-b072-da95cddd2379',
 'url': 'https://www.neb-one.gc.ca/open/energy/energyfutures2016/primary-energy-demand-2016.csv',
 'format': 'CSV',
 'description': '',
 'hash': '',
 'position': 60,
 'name': 'Primary Energy Demand 2016',
 'resource_type': 'dataset',
 'mimetype': None,
 'mimetype_inner': None,
 'size': None,
 'created': '2019-02-27T21:40:46.957239',
 'last_modified': None,
 'metadata_modified': '2019-02-27T21:40:46.957239',
 'cache_url': None,
 'cache_last_updated': None,
 'url_type': None,
 'state': 'active',
 'related_type': '',
 'name_translated': '{"fr": "demande-d-energie-primaire-2016", "en": "Primary Energy Demand 2016"}',
 'language': '["en"]',
 'unique_identifier': '',
 'date_published': '',
 'related_relationship': '',
 'datastore_active': False,
 'character_set': '',
 'data_quality': '[]',
 'schema': ['sector',
  'case',
  'region',
  'variable_english',
  'year',
  'value',
  'sector_1']}

In [30]:
remote_table = pd.read_csv(md['url'], encoding_errors='ignore')
remote_table

,Sector,Case,Region,Variable_English,Year,Value,Sector.1
0,Electric and Steam Generation,Reference,Canada,Natural Gas,2005,401.8711,Electric and Steam Generation
1,Electric and Steam Generation,Reference,Canada,Natural Gas,2006,432.9016,Electric and Steam Generation
2,Electric and Steam Generation,Reference,Canada,Natural Gas,2007,434.1760,Electric and Steam Generation
3,Electric and Steam Generation,Reference,Canada,Natural Gas,2008,458.2970,Electric and Steam Generation
4,Electric and Steam Generation,Reference,Canada,Natural Gas,2009,437.5902,Electric and Steam Generation
...,...,...,...,...,...,...,...
42331,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2036,0.3401,Primary Demand
42332,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2037,0.3426,Primary Demand
42333,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2038,0.3452,Primary Demand
42334,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2039,0.3385,Primary Demand


In [ ]:
import zipfile

local_table = None

# for each table which is a candidate for a mapping with 
# the current remote table, look if the two are identical
with zipfile.ZipFile(f'{data_path}/datasets/datasets_CAN.zip') as z:
    for candidate in candidates[tid]:
        with z.open(f'datasets_CAN/{candidate}') as fr:
            table = pd.read_csv(fr)
            print(table.shape)
            if table.equals(remote_table):
                print('found ', candidate, table.equals(remote_table))
                local_table = table
local_table

(42336, 7)
found  CAN_CSV0000000000004354.csv True
(42336, 7)
found  CAN_CSV0000000000016880.csv True


,Sector,Case,Region,Variable_English,Year,Value,Sector.1
0,Electric and Steam Generation,Reference,Canada,Natural Gas,2005,401.8711,Electric and Steam Generation
1,Electric and Steam Generation,Reference,Canada,Natural Gas,2006,432.9016,Electric and Steam Generation
2,Electric and Steam Generation,Reference,Canada,Natural Gas,2007,434.1760,Electric and Steam Generation
3,Electric and Steam Generation,Reference,Canada,Natural Gas,2008,458.2970,Electric and Steam Generation
4,Electric and Steam Generation,Reference,Canada,Natural Gas,2009,437.5902,Electric and Steam Generation
...,...,...,...,...,...,...,...
42331,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2036,0.3401,Primary Demand
42332,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2037,0.3426,Primary Demand
42333,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2038,0.3452,Primary Demand
42334,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2039,0.3385,Primary Demand


In [32]:
table

,Sector,Case,Region,Variable_English,Year,Value,Sector.1
0,Electric and Steam Generation,Reference,Canada,Natural Gas,2005,401.8711,Electric and Steam Generation
1,Electric and Steam Generation,Reference,Canada,Natural Gas,2006,432.9016,Electric and Steam Generation
2,Electric and Steam Generation,Reference,Canada,Natural Gas,2007,434.1760,Electric and Steam Generation
3,Electric and Steam Generation,Reference,Canada,Natural Gas,2008,458.2970,Electric and Steam Generation
4,Electric and Steam Generation,Reference,Canada,Natural Gas,2009,437.5902,Electric and Steam Generation
...,...,...,...,...,...,...,...
42331,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2036,0.3401,Primary Demand
42332,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2037,0.3426,Primary Demand
42333,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2038,0.3452,Primary Demand
42334,Primary Demand,Constrained,Yukon,Other Renewables and Landfill Gas,2039,0.3385,Primary Demand


In [34]:
# if the remote table and the found local table are identical, this should result into an empty data frame
remote_table[~remote_table.apply(tuple, 1).isin(local_table.apply(tuple, 1))]

,Sector,Case,Region,Variable_English,Year,Value,Sector.1
